# Name That Model - Training Pipeline

This notebook trains a model to identify the authorship of AI-generated text passages among 18 candidate frontier language models.

The task: Given a text passage, predict which of the 18 models generated it.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, balanced_accuracy_score
import re
import json
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded')

## 1. Load Training Data

In [ ]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(f'Training samples: {len(train)}')
print(f'Test samples: {len(test)}')
print('\nModel distribution:')
print(train['model'].value_counts())

## 2. Text Preprocessing

In [ ]:
def clean_text(text):
    """Basic text cleaning: lowercase, remove punctuation, collapse whitespace."""
    text = text.lower()
    text = re.sub(r'["'"'"]"', '', text)  # remove quotes
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train['clean_text'] = train['text'].astype(str).apply(clean_text)
test['clean_text'] = test['text'].astype(str).apply(clean_text)

print(f"Cleaned training samples: {len(train)}")
print(f"Cleaned test samples: {len(test)}")

## 3. Feature Engineering & Model Training

In [ ]:
# Extract linguistic features from text
def extract_features(text_series):
    features = pd.DataFrame(index=text_series.index)
    text = text_series.astype(str)
    
    # Basic text statistics
    features['char_len'] = text.str.len()
    features['word_count'] = text.apply(lambda x: len(x.split()) if pd.notna(x) else 0)
    features['sentence_count'] = text.apply(lambda x: len([s for s in str(x).split('.') if s.strip()]) if pd.notna(x) else 0)
    features['avg_word_len'] = features['char_len'] / (features['word_count'] + 1e-5)
    
    # Lexical richness (type-token ratio)
    def ttr(series):
        result = {}
        for idx, val in series.items():
            words = str(val).lower().split()
            if len(words) > 0:
                result[idx] = len(set(words)) / len(words)
            else:
                result[idx] = 0
        return pd.Series(result, name='ttr')
    features['ttr'] = ttr(text)
    
    # Long word ratio (indicative of technical/verbose writing)
    def long_word_ratio(series):
        result = {}
        for idx, val in series.items():
            words = str(val).lower().split()
            long_words = [w for w in words if len(w) > 10]
            result[idx] = len(long_words) / len(words) if len(words) > 0 else 0
        return pd.Series(result, name='long_word_ratio')
    features['long_word_ratio'] = long_word_ratio(text)
    
    # Average sentence length
    def sent_len(series):
        result = {}
        for idx, val in series.items():
            sents = [s for s in str(val).split('.') if s.strip()]
            avg = np.mean([len(s.split()) for s in sents]) if len(sents) > 0 else 0
            result[idx] = avg
        return pd.Series(result, name='avg_sent_len')
    features['avg_sent_len'] = sent_len(text)
    
    return features.fillna(0)

train_features = extract_features(train['clean_text'])
test_features = extract_features(test['clean_text'])

# Reindex to include all feature columns
feature_cols = list(train_features.columns)
train_final = pd.concat([train.reset_index(drop=True), train_features.reset_index(drop=True)], axis=1)
test_final = pd.concat([test.reset_index(drop=True), test_features.reset_index(drop=True)], axis=1)

print(f'Feature matrix shape: {train_final.shape}')
print(f'Feature columns: {feature_cols}')

## 4. Train Classifier

In [ ]:
# Encode labels
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(train['model'])

# Check class balance
class_counts = train['model'].value_counts()
print(f'Number of classes: {len(class_counts)}')
print(f'Class balance:\n{class_counts}')

# Build pipeline with TF-IDF + Logistic Regression
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 2),
        stop_words='english'
    )),
    ('clf', LogisticRegression(
        max_iter=1000,
        multi_class='auto',
        solver='lbfgs'
    ))
])

# Train
print('Training classifier...')
pipeline.fit(train['clean_text'], encoded_labels)
print('Training complete')

## 5. Evaluate on Training Set

In [ ]:
# Predict on training set
train_preds = pipeline.predict(train['clean_text'])
train_acc = accuracy_score(train_preds, encoded_labels)
print(f'Training accuracy: {train_acc:.4f}')

# Balanced accuracy
bal_acc = balanced_accuracy_score(encoded_labels, train_preds)
print(f'Balanced accuracy: {bal_acc:.4f}')

results = classification_report(encoded_labels, train_preds, target_names=label_encoder.classes_)
print(results)

## 6. Predict on Test Set & Create Submission

In [ ]:
# Predict on test data
test_preds = pipeline.predict(test['clean_text'])
test_labels = label_encoder.inverse_transform(test_preds)

# Create submission dataframe
submission = pd.DataFrame({
    'id': test['id'].astype(str),
    'model': test_labels
})

# Ensure model names are valid
valid_models = [
    'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
    'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
    'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5',
    'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3'
]
submission['model'] = submission['model'].apply(lambda x: x if x in valid_models else 'gpt-5.5')

# Save submission
submission.to_parquet('submission.parquet', index=False)
submission.to_csv('submission.csv', index=False)

print('Submission created: submission.parquet')
print(submission.head(10))